In [6]:
import sys, os
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
os.chdir(PROJECT_ROOT)

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

In [38]:
PROJECT_ROOT

PosixPath('/home/jovyan/project/supplementary/LLM-TSAD')

In [7]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import time
import re
import json
import math
import glob
from scipy.signal import find_peaks
from statsmodels.tsa.seasonal import seasonal_decompose

from datetime import datetime, timedelta

In [8]:
pd.set_option('display.max_colwidth', None)

In [9]:
from prompt import time_series_to_image
from utils import view_base64_image, display_messages, collect_results, compute_metrics, interval_to_vector, plot_series_and_predictions
from data.synthetic import SyntheticDataset

### Run all

In [39]:
def average_dict_values(dict_list):
    sums = {}
    counts = {}

    for d in dict_list:
        for key, value in d.items():
            sums[key] = sums.get(key, 0) + value
            counts[key] = counts.get(key, 0) + 1
            
    # 각 키에 대해 평균 계산: 합계 / 등장횟수
    averages = {key: sums[key] / counts[key] for key in sums}
    return averages

In [95]:
aaa = sorted(glob.glob('./results/synthetic/*/gemini-1.5-flash/0shot-text-vision.jsonl'))
aaa += sorted(glob.glob('./results/synthetic/*/gpt-4o/0shot-text-vision.jsonl'))
aaa

['./results/synthetic/freq/gemini-1.5-flash/0shot-text-vision.jsonl',
 './results/synthetic/point/gemini-1.5-flash/0shot-text-vision.jsonl',
 './results/synthetic/range/gemini-1.5-flash/0shot-text-vision.jsonl',
 './results/synthetic/trend/gemini-1.5-flash/0shot-text-vision.jsonl',
 './results/synthetic/freq/gpt-4o/0shot-text-vision.jsonl',
 './results/synthetic/point/gpt-4o/0shot-text-vision.jsonl',
 './results/synthetic/range/gpt-4o/0shot-text-vision.jsonl',
 './results/synthetic/trend/gpt-4o/0shot-text-vision.jsonl']

In [96]:
all_results = []
for path in aaa:
    result_df = pd.read_json(path, lines=True)
    print(path, result_df.shape)
    a =  average_dict_values(result_df['metrics'].tolist())
    
    log_str = []
    for k, v in a.items():
#         print(v)
        log_str.append(f'{v*100:.2f}')
    print('\t'.join(log_str))
    
#     for k, v in a.items():
#         print(f"{k}: {v*100:.2f}")
    all_results.append(a)
    print()

./results/synthetic/freq/gemini-1.5-flash/0shot-text-vision.jsonl (400, 5)
48.58	48.57	41.78	82.75	70.73	73.84

./results/synthetic/point/gemini-1.5-flash/0shot-text-vision.jsonl (400, 5)
90.30	74.08	77.49	98.08	96.83	96.98

./results/synthetic/range/gemini-1.5-flash/0shot-text-vision.jsonl (400, 5)
53.34	59.94	54.06	66.36	68.37	67.12

./results/synthetic/trend/gemini-1.5-flash/0shot-text-vision.jsonl (400, 5)
83.67	81.88	81.74	90.12	92.50	91.05

./results/synthetic/freq/gpt-4o/0shot-text-vision.jsonl (400, 5)
51.33	30.80	33.68	64.46	47.78	52.50

./results/synthetic/point/gpt-4o/0shot-text-vision.jsonl (400, 5)
96.66	69.79	77.59	97.74	94.11	95.39

./results/synthetic/range/gpt-4o/0shot-text-vision.jsonl (400, 5)
96.32	88.56	91.58	98.09	96.37	96.97

./results/synthetic/trend/gpt-4o/0shot-text-vision.jsonl (400, 5)
72.90	71.82	72.02	73.96	73.91	73.94



In [94]:
a = average_dict_values(all_results)
log_str = []
for k, v in a.items():
#         print(v)
    log_str.append(f'{v*100:.2f}')
print('\t'.join(log_str))
    

68.97	66.12	63.77	84.33	82.11	82.25


In [84]:
raise Exception('')

Exception: 

In [212]:
copy_files_to_folder(aaa, './neurips_table1/gpt-4o/')

[복사] ./src/results/trend_gpt-4o_itype-number_ar5_maxlen2000_ts400_imageTrue_dsFalse.jsonl → ./neurips_table1/gpt-4o/trend_gpt-4o_itype-number_ar5_maxlen2000_ts400_imageTrue_dsFalse.jsonl
[복사] ./src/results/range_gpt-4o_itype-number_ar5_maxlen2000_ts400_imageTrue_dsFalse.jsonl → ./neurips_table1/gpt-4o/range_gpt-4o_itype-number_ar5_maxlen2000_ts400_imageTrue_dsFalse.jsonl
[복사] ./src/results/point_gpt-4o_itype-number_ar5_maxlen2000_ts400_imageTrue_dsFalse.jsonl → ./neurips_table1/gpt-4o/point_gpt-4o_itype-number_ar5_maxlen2000_ts400_imageTrue_dsFalse.jsonl
[복사] ./src/results/freq_gpt-4o_itype-number_ar5_maxlen2000_ts400_imageTrue_dsFalse.jsonl → ./neurips_table1/gpt-4o/freq_gpt-4o_itype-number_ar5_maxlen2000_ts400_imageTrue_dsFalse.jsonl


In [27]:
data_name = 'TODS'
model_name = 'time'
aaa = glob.glob(f'./src/results/{data_name}*-{model_name}*.jsonl')
aaa

['./src/results/TODS_gpt-4o_itype-timestamp_ar50_maxlen2000_ts13.jsonl']

In [53]:
all_results = []
for path in aaa:
    result_df = pd.read_json(path, lines=True)
    print(path, result_df.shape)
    a =  average_dict_values(result_df['metric'].tolist())
    a['inference_time'] = result_df['inference_time'].mean()
    
    log_str = []
    for k, v in a.items():
#         print(v)
        log_str.append(f'{v:.4f}')
    print('\t'.join(log_str))
    
#     for k, v in a.items():
#         print(f"{k}: {v*100:.2f}")
    all_results.append(a)
    print()



./src/results/TODS_gpt-4o_itype-timestamp_ar50_maxlen2000_ts13.jsonl (13, 4)
0.7177	0.3549	0.4645	0.9487	0.5318	0.6672	43.1657



In [252]:
thresholds = dict() 
thresholds['trend'] = {'1': 57.5, '2': 57.5, '3': 57.5, '4': 57.5, '5': 57.5, '6': 57.5}
thresholds['freq'] = {'1': 10, '2': 10, '3': 10, '4': 10, '5': 10, '6': 10}
thresholds['point'] = {'1': 29.25, '2': 29.25, '3': 29.25, '4': 29.25, '5': 29.25, '6': 29.25}
thresholds['range'] = {'1': 30.25, '2': 30.25, '3': 30.25, '4': 30.25, '5': 30.25, '6': 30.25}

In [257]:
baseline = average_dict_values([v for k, v in thresholds.items()])
baseline

{'1': 31.75, '2': 31.75, '3': 31.75, '4': 31.75, '5': 31.75, '6': 31.75}

In [263]:
a = average_dict_values(all_results)
log_str = []
diffs = []
for i, (k, v) in enumerate(a.items()):
#         print(v)
    base_val = baseline[str(i+1)] 
    log_str.append(f'{v*100:.2f}')
    diffs.append(v*100 - base_val)
print('\t'.join(log_str))
print(f'{np.mean(diffs[:3]):.2f}\t{np.mean(diffs[3:]):.2f}')
    

76.94	68.42	70.03	83.10	78.85	80.11
40.05	48.94


In [54]:
aaa = glob.glob('*mini*vision-True_results*.jsonl')
aaa

['freq_gpt-4o-mini_ts400_vision-True_results3.jsonl',
 'range_gpt-4o-mini_ts400_vision-True_results3.jsonl',
 'trend_gpt-4o-mini_ts400_vision-True_results3.jsonl',
 'point_gpt-4o-mini_ts400_vision-True_results3.jsonl']

In [55]:
all_results = []
for path in aaa:
    result_df = pd.read_json(path, lines=True)
    print(path, result_df.shape)
    a =  average_dict_values(result_df['metric'].tolist())
    for k, v in a.items():
        print(f"{k}: {v*100:.2f}")
    all_results.append(a)
    print()
    
all_results

freq_gpt-4o-mini_ts400_vision-True_results3.jsonl (400, 6)
precision: 13.64
recall: 22.15
f1: 15.03
affi precision: 29.09
affi recall: 29.46
affi f1: 28.26

range_gpt-4o-mini_ts400_vision-True_results3.jsonl (400, 6)
precision: 38.41
recall: 46.07
f1: 39.79
affi precision: 51.10
affi recall: 53.21
affi f1: 51.54

trend_gpt-4o-mini_ts400_vision-True_results3.jsonl (400, 6)
precision: 35.37
recall: 40.83
f1: 36.63
affi precision: 44.01
affi recall: 47.51
affi f1: 45.26

point_gpt-4o-mini_ts400_vision-True_results3.jsonl (400, 6)
precision: 43.16
recall: 66.52
f1: 47.01
affi precision: 68.21
affi recall: 74.11
affi f1: 69.70



[{'precision': 0.13638250000000005,
  'recall': 0.2215375,
  'f1': 0.1502875,
  'affi precision': 0.29094249999999994,
  'affi recall': 0.2945675000000001,
  'affi f1': 0.28255250000000015},
 {'precision': 0.38410750000000005,
  'recall': 0.4607425,
  'f1': 0.3979249999999999,
  'affi precision': 0.5110175,
  'affi recall': 0.53214,
  'affi f1': 0.5154025000000001},
 {'precision': 0.35365250000000004,
  'recall': 0.4083075000000001,
  'f1': 0.366255,
  'affi precision': 0.44006999999999996,
  'affi recall': 0.47512999999999983,
  'affi f1': 0.45264250000000006},
 {'precision': 0.43162999999999996,
  'recall': 0.665225,
  'f1': 0.4700625,
  'affi precision': 0.6820925000000001,
  'affi recall': 0.7411274999999996,
  'affi f1': 0.6969825000000003}]

In [21]:
all_results[0]


{'precision': 0.5512749999999995,
 'recall': 0.8793675000000001,
 'f1': 0.6292800000000001,
 'affi precision': 0.9022800000000005,
 'affi recall': 0.9152175000000007,
 'affi f1': 0.9047225000000002}

In [ ]:
all_results[0]


In [13]:
import pandas as pd
from io import StringIO

# Multiline string input
a = """Datasets	Code	Prompt	1	2	3	4	5	6
Trend	A	1shot-vision-cot	54.59	55.00	54.66	54.94	55.00	54.97
	B	1shot-vision-calc	43.78	44.79	42.79	48.13	48.69	48.37
	C	1shot-vision-dyscalc	42.98	46.53	42.82	49.49	50.76	50.03
	E	0shot-vision-cot	55.93	55.89	55.81	56.19	56.22	56.20
	F	0shot-vision-calc	61.30	60.87	60.69	62.07	62.06	62.06
	G	0shot-vision-dyscalc	60.75	60.74	60.43	61.56	61.62	61.58
	H	0shot-vision	60.19	59.95	59.79	60.64	60.64	60.64
	J	1shot-text-s0.3	45.52	45.51	45.52	47.48	47.99	47.69
	L	0shot-text-s0.3-pap	40.50	40.50	40.50	40.76	40.84	40.79
	M	0shot-text-s0.3-dyscalc	10.25	10.25	10.25	10.98	11.38	11.13
	N	0shot-text-s0.3-csv	16.75	16.75	16.75	19.08	20.10	19.44
	P	0shot-text-s0.3-cot-pap	18.75	18.75	18.75	19.01	19.15	19.06
	S	0shot-text-s0.3-calc	8.81	8.80	8.81	10.03	10.88	10.33
	T	0shot-text-s0.3	46.81	46.77	46.78	48.76	49.48	49.02
	U	0shot-text	56.00	56.00	56.00	56.08	56.14	56.11
	Our	0shot-text	86.06	83.24	82.00	91.96	93.96	92.69
	Our	0shot-text-vision (w/o value)	61.38	60.66	60.77	62.70	62.86	62.77
	Our	0shot-text-vision (w/o index)	66.82	68.73	66.34	81.77	84.62	82.88
	Our	0shot-text-vision (w/o deseason)	60.80	59.36	59.68	61.37	61.27	61.31
	Our	0shot-text-vision	85.61	81.49	81.09	89.95	91.19	90.20
Freq	A	1shot-vision-cot	5.16	7.29	5.35	16.12	14.46	14.67
	B	1shot-vision-calc	10.98	11.84	10.00	28.72	23.05	23.95
	C	1shot-vision-dyscalc	10.15	11.80	9.17	31.71	26.34	26.81
	E	0shot-vision-cot	9.33	13.26	9.93	17.26	16.61	16.33
	F	0shot-vision-calc	14.29	15.52	14.15	31.78	24.87	26.59
	G	0shot-vision-dyscalc	16.40	18.10	16.39	31.99	24.42	26.38
	H	0shot-vision	15.47	18.62	15.75	29.05	26.55	26.64
	J	1shot-text-s0.3	16.39	16.31	15.45	63.17	50.72	54.66
	L	0shot-text-s0.3-pap	6.75	6.75	6.75	7.98	7.36	7.52
	M	0shot-text-s0.3-dyscalc	3.72	3.56	3.50	19.20	14.15	15.51
	N	0shot-text-s0.3-csv	11.16	9.81	9.98	33.34	23.73	26.37
	P	0shot-text-s0.3-cot-pap	4.81	4.82	4.81	5.26	5.06	5.12
	S	0shot-text-s0.3-calc	5.51	4.96	4.76	19.25	13.76	15.27
	T	0shot-text-s0.3	19.30	18.79	18.14	64.73	52.41	56.38
	U	0shot-text	11.45	10.11	10.20	18.80	14.19	15.43
	Our	0shot-text	56.91	30.77	33.57	80.90	62.84	67.54
	Our	0shot-text-vision (w/o value)	23.87	28.34	22.97	61.32	51.76	53.90
	Our	0shot-text-vision (w/o index)	21.97	27.84	21.22	63.38	53.50	55.76
	Our	0shot-text-vision (w/o deseason)	42.81	28.58	31.05	47.43	37.21	40.30
	Our	0shot-text-vision	49.41	41.14	37.88	79.18	64.95	68.57
Point	A	1shot-vision-cot	18.47	22.57	18.98	35.89	38.85	36.88
	B	1shot-vision-calc	41.02	43.32	40.04	73.97	79.19	75.82
	C	1shot-vision-dyscalc	37.43	42.19	37.49	70.08	75.13	71.84
	E	0shot-vision-cot	25.04	32.84	26.86	44.71	46.66	45.33
	F	0shot-vision-calc	46.25	51.14	46.36	82.45	83.61	82.77
	G	0shot-vision-dyscalc	47.30	53.39	47.98	82.86	84.52	83.41
	H	0shot-vision	60.09	67.56	61.62	94.93	96.19	95.34
	J	1shot-text-s0.3	33.19	31.96	32.18	70.90	65.03	66.95
	L	0shot-text-s0.3-pap	16.25	16.25	16.25	18.21	17.68	17.83
	M	0shot-text-s0.3-dyscalc	12.62	12.19	12.27	26.96	23.93	24.80
	N	0shot-text-s0.3-csv	18.06	16.77	17.03	39.32	34.06	35.67
	P	0shot-text-s0.3-cot-pap	8.50	8.50	8.50	8.85	8.70	8.75
	S	0shot-text-s0.3-calc	6.61	6.36	6.41	18.27	15.88	16.57
	T	0shot-text-s0.3	29.66	29.23	29.16	68.38	62.84	64.63
	U	0shot-text	30.33	29.85	29.83	62.56	54.68	57.09
	Our	0shot-text	94.52	56.07	63.16	98.31	91.50	93.78
	Our	0shot-text-vision (w/o value)	56.83	58.55	55.57	92.67	90.89	91.23
	Our	0shot-text-vision (w/o index)	52.61	54.97	52.19	89.04	89.42	88.50
	Our	0shot-text-vision (w/o deseason)	90.61	95.63	91.74	99.36	99.53	99.36
	Our	0shot-text-vision	94.19	83.80	85.98	99.01	96.94	97.70
Range	A	1shot-vision-cot	30.14	31.44	29.92	51.12	51.95	51.00
	B	1shot-vision-calc	49.75	51.22	48.58	94.28	91.52	92.25
	C	1shot-vision-dyscalc	49.84	51.61	48.97	94.03	90.73	91.68
	E	0shot-vision-cot	36.46	43.52	37.61	61.41	61.55	61.27
	F	0shot-vision-calc	46.03	55.76	48.02	92.50	91.52	91.57
	G	0shot-vision-dyscalc	46.97	58.48	49.17	91.06	91.02	90.67
	H	0shot-vision	47.35	57.86	49.43	93.36	92.56	92.49
	J	1shot-text-s0.3	14.26	16.71	14.76	50.94	55.36	52.01
	L	0shot-text-s0.3-pap	9.65	9.30	9.39	22.67	20.76	21.09
	M	0shot-text-s0.3-dyscalc	2.35	2.30	2.31	8.47	7.66	7.84
	N	0shot-text-s0.3-csv	9.77	9.14	9.20	32.94	28.68	29.93
	P	0shot-text-s0.3-cot-pap	3.94	3.33	3.40	6.23	5.32	5.52
	S	0shot-text-s0.3-calc	0.69	0.56	0.59	3.37	2.92	3.03
	T	0shot-text-s0.3	11.42	13.58	11.85	46.77	51.10	47.91
	U	0shot-text	6.19	6.68	5.93	45.32	44.74	43.80
	Our	0shot-text	52.40	46.27	45.78	60.49	65.31	61.84
	Our	0shot-text-vision (w/o value)	47.70	53.44	48.04	91.94	89.46	90.06
	Our	0shot-text-vision (w/o index)	13.26	22.41	15.17	53.77	61.26	56.37
	Our	0shot-text-vision (w/o deseason)	61.49	66.94	62.43	70.08	73.02	71.26
	Our	0shot-text-vision	59.79	67.18	61.21	69.91	73.02	71.03"""

# Create DataFrame
df = pd.read_csv(StringIO(a), sep="\t")
df.Datasets = df.Datasets.ffill()

In [14]:
# 2. 각 열에 대한 임계값(특정 수치)을 미리 정의 (사용자 필요에 따라 조정)
thresholds = dict() 
thresholds['trend'] = {'1': 57.5, '2': 57.5, '3': 57.5, '4': 57.5, '5': 57.5, '6': 57.5}
thresholds['freq'] = {'1': 10, '2': 10, '3': 10, '4': 10, '5': 10, '6': 10}
thresholds['point'] = {'1': 29.25, '2': 29.25, '3': 29.25, '4': 29.25, '5': 29.25, '6': 29.25}
thresholds['range'] = {'1': 30.25, '2': 30.25, '3': 30.25, '4': 30.25, '5': 30.25, '6': 30.25}


def generate_latex_table(
    df: pd.DataFrame,
    thresholds: dict,
    dataset_col: str = 'Datasets',
    code_col: str = 'Code',
    prompt_col: str = 'Prompt',
    min_saturation: int = 50  # 최소 채도 값 (0–100)
) -> str:

    metric_cols = [c for c in df.columns if c not in {dataset_col, code_col, prompt_col}]

    def text_color(val, thr):
        diff = val - thr
        ratio = abs(diff) / 100 if thr else 0
        pct = min(int(ratio * 100), 100)
        if pct < min_saturation:
            pct = min_saturation
        return f'green!{pct}!black' if diff > 0 else f'red!{pct}!black'

    lines = []
    # 헤더
    lines.append(f"{dataset_col} & {code_col} & {prompt_col} & " +
                 " & ".join(metric_cols) + r" \\")
    lines.append(r"\midrule")

    row_counter = 0
    for ds, grp in df.groupby(dataset_col, sort=False):
        ds_key = ds.lower()
        nrows = len(grp)

        # 이 데이터셋 그룹에서 “Our” 구분선 삽입용 플래그
        our_encountered = False

        # 그룹 내 최대값 & 두 번째 최대값 계산
        max_vals = grp[metric_cols].max()
        second_max_vals = {
            col: (sorted(grp[col].unique(), reverse=True)[1]
                  if len(grp[col].unique()) > 1 else grp[col].unique()[0])
            for col in metric_cols
        }

        for idx, (_, row) in enumerate(grp.iterrows()):
            # 이 그룹에서 “Our”가 처음 나올 때마다 cmidrule 삽입
            if not our_encountered and str(row[code_col]) == "Our":
                lines.append(r"\cmidrule{2-9}")
                our_encountered = True

            shaded = (row_counter % 2 == 1)
            cells = []

            # multirow for dataset
            if idx == 0:
                cells.append(rf'\multirow{{{nrows}}}{{*}}{{{ds}}}')
            else:
                cells.append('{}')

            # code & prompt
            for col in (code_col, prompt_col):
                txt = str(row[col])
                if shaded:
                    txt = rf'\cellcolor{{gray!15}}{txt}'
                cells.append(txt)

            # metrics 컬럼들
            for col in metric_cols:
                val = row[col]
                thr = thresholds[ds_key][col]
                formatted = rf'\textcolor{{{text_color(val, thr)}}}{{{val:.2f}}}'
                if val == max_vals[col]:
                    formatted = rf'\textbf{{{formatted}}}'
                elif val == second_max_vals[col]:
                    formatted = rf'\underline{{{formatted}}}'
                if shaded:
                    formatted = rf'\cellcolor{{gray!15}}{formatted}'
                cells.append(formatted)

            lines.append(" & ".join(cells) + r" \\")
            row_counter += 1

        # 그룹마다 midrule
        lines.append(r"\midrule")

    # 마지막 midrule을 bottomrule로 대체
    lines[-1] = r"\bottomrule"
    return "\n".join(lines)

latex = generate_latex_table(df, thresholds)
print(latex)

Datasets & Code & Prompt & 1 & 2 & 3 & 4 & 5 & 6 \\
\midrule
\multirow{20}{*}{Trend} & A & 1shot-vision-cot & \textcolor{red!50!black}{54.59} & \textcolor{red!50!black}{55.00} & \textcolor{red!50!black}{54.66} & \textcolor{red!50!black}{54.94} & \textcolor{red!50!black}{55.00} & \textcolor{red!50!black}{54.97} \\
{} & \cellcolor{gray!15}B & \cellcolor{gray!15}1shot-vision-calc & \cellcolor{gray!15}\textcolor{red!50!black}{43.78} & \cellcolor{gray!15}\textcolor{red!50!black}{44.79} & \cellcolor{gray!15}\textcolor{red!50!black}{42.79} & \cellcolor{gray!15}\textcolor{red!50!black}{48.13} & \cellcolor{gray!15}\textcolor{red!50!black}{48.69} & \cellcolor{gray!15}\textcolor{red!50!black}{48.37} \\
{} & C & 1shot-vision-dyscalc & \textcolor{red!50!black}{42.98} & \textcolor{red!50!black}{46.53} & \textcolor{red!50!black}{42.82} & \textcolor{red!50!black}{49.49} & \textcolor{red!50!black}{50.76} & \textcolor{red!50!black}{50.03} \\
{} & \cellcolor{gray!15}E & \cellcolor{gray!15}0shot-vision-co